# Parsers against each other and against the judge

Runs the current Python parser, the Scala `PeriodParser` (through `PeriodParserCli` and sbt) and
`period_new` over every string in `data/periods.jsonl`, shows where they disagree, and scores each
against the judge's answers in `data/judge-output`. Ranges are compared as inclusive day bounds
with open sides as `None`; identifiers are not compared.

In [1]:
import json
import subprocess
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

from adapters.transformers.marc.parsers.period import parse as parse_new
from period_old import old_bounds

rows = [json.loads(line) for line in Path("data/periods.jsonl").open(encoding="utf-8")]
occurrences = Counter()
for row in rows:
    occurrences[(row["source"], row["path"], row["text"])] += row["n"]
print(f"{sum(occurrences.values())} strings, {len(occurrences)} distinct (source, path, text) triples")

1547871 strings, 76343 distinct (source, path, text) triples


In [2]:
OPEN = {"0001-01-01", "9999-12-31", "-9999-01-01"}


def bounds(start, end):
    return tuple(None if not d or d[:10] in OPEN else d[:10] for d in (start, end))


def python_bounds(source, path, text):
    rng = old_bounds(path, text)
    return bounds(*rng) if rng else (None, None)


def new_bounds(source, path, text):
    span = parse_new(text, source)
    return bounds(span[0].isoformat(), span[1].isoformat()) if span else (None, None)

In [3]:
scala_in = Path("data/scala-in.txt").resolve()
scala_out = Path("data/scala-out.jsonl").resolve()
scala_in.write_text("\n".join(sorted({text.replace("\n", " ") for _, _, text in occurrences})), encoding="utf-8")
sbt = subprocess.run(
    ["sbt", "-batch", f"transformer_common/Test/runMain weco.pipeline.transformer.parse.PeriodParserCli {scala_in} {scala_out}"],
    cwd="../..", capture_output=True, text=True,
)
assert sbt.returncode == 0, sbt.stdout[-2000:] + sbt.stderr[-2000:]

scala = {}
for line in scala_out.read_text(encoding="utf-8").splitlines():
    row = json.loads(line)
    rng = row["range"] or {}
    scala[row["input"]] = bounds(rng.get("from"), rng.get("to"))

PARSERS = {"python": python_bounds, "scala": lambda source, path, text: scala[text], "new": new_bounds}
results = {name: {key: parser(*key) for key in occurrences} for name, parser in PARSERS.items()}

In [4]:
def verdict(key):
    got = {name: results[name][key] for name in PARSERS}
    if len(set(got.values())) == 1:
        return "all agree"
    return "differ: " + ", ".join(sorted(name for name in PARSERS if list(got.values()).count(got[name]) == 1))


verdicts = {key: verdict(key) for key in occurrences}
summary = Counter()
for key, v in verdicts.items():
    summary[(key[1], v)] += occurrences[key]
for (path, v), n in sorted(summary.items()):
    print(f"{path:11} {v:24} {n:>9}")

genre       all agree                     7407
genre       differ: new                      2
genre       differ: new, python, scala         4
genre       differ: python                 954
production  all agree                  1258290
production  differ: new                   2930
production  differ: new, python, scala      7802
production  differ: python               86968
production  differ: scala                15117
subject     all agree                   132900
subject     differ: new                  35257
subject     differ: new, python, scala        73
subject     differ: python                 157
subject     differ: scala                   10


In [5]:
LIMIT = 40


def show(b):
    return f"{b[0] or 'open'} .. {b[1] or 'open'}" if b != (None, None) else "no range"


for key in sorted((k for k in occurrences if verdicts[k] != "all agree"), key=lambda k: -occurrences[k])[:LIMIT]:
    source, path, text = key
    print(f"{text!r}  [{source} {path}, {occurrences[key]}x]")
    for name in PARSERS:
        print(f"    {name:7} {show(results[name][key])}")

'19th-20th centuries.'  [marc subject, 6934x]
    python  no range
    scala   no range
    new     1800-01-01 .. 1999-12-31
'18th-19th centuries.'  [marc subject, 2577x]
    python  no range
    scala   no range
    new     1700-01-01 .. 1899-12-31
'To 1500.'  [marc subject, 1784x]
    python  no range
    scala   no range
    new     open .. 1500-12-31
'Revolution, 1775-1783'  [marc subject, 1510x]
    python  no range
    scala   no range
    new     1775-01-01 .. 1783-12-31
'16th-17th centuries.'  [marc subject, 1196x]
    python  no range
    scala   no range
    new     1500-01-01 .. 1699-12-31
'17th-18th centuries.'  [marc subject, 1117x]
    python  no range
    scala   no range
    new     1600-01-01 .. 1799-12-31
'18th-20th centuries.'  [marc subject, 891x]
    python  no range
    scala   no range
    new     1700-01-01 .. 1999-12-31
'Revolution, 1775-1783.'  [marc subject, 821x]
    python  no range
    scala   no range
    new     1775-01-01 .. 1783-12-31
'Revolution, 1789

## Against the judge

`unparseable` and `ambiguous` both mean no range is right. Judge answers that are malformed or
run backwards are dropped. Precision is correct ranges over ranges produced; recall is correct
ranges over ranges the judge found.

In [6]:
index = json.load(Path("data/judge-input/index.json").open(encoding="utf-8"))
judged, malformed = {}, 0
for f in sorted(Path("data/judge-output").glob("batch-*.jsonl")):
    for line in f.read_text(encoding="utf-8").splitlines():
        if not line.strip() or line.startswith("```"):
            continue
        try:
            id_, *rest = json.loads(line)
            assert len(rest) == 5 and id_ in index
            judged[id_] = tuple(rest)
        except (ValueError, AssertionError):
            malformed += 1


def valid(outcome, start, end):
    if outcome != "range":
        return outcome in ("unparseable", "ambiguous")
    try:
        return bool(start or end) and all(date.fromisoformat(d.lstrip("-")) for d in (start, end) if d) and (not start or not end or start <= end)
    except ValueError:
        return False


expected = {}
for id_, (outcome, start, end, qualifier, note) in judged.items():
    if valid(outcome, start, end):
        for path in index[id_]["paths"]:
            expected[(id_, path)] = bounds(start, end) if outcome == "range" else (None, None)

got = {name: {(id_, path): results[name][(index[id_]["source"], path, index[id_]["text"])] for id_, path in expected} for name in PARSERS}
weight = lambda key: occurrences[(index[key[0]]["source"], key[1], index[key[0]]["text"])]

print(f"{len(judged)} judged, {malformed} malformed lines skipped, {len(judged) - len({k[0] for k in expected})} dropped as invalid\n")
print(f"{'parser':8} {'correct':>8} {'wrong':>6}   {'accuracy':>9} {'precision':>10} {'recall':>7}   {'weighted acc':>12} {'prec':>6} {'recall':>7}")
for name in PARSERS:
    line = f"{name:8}"
    for w in (lambda key: 1, weight):
        total = sum(w(k) for k in expected)
        correct = sum(w(k) for k in expected if got[name][k] == expected[k])
        produced = sum(w(k) for k in expected if got[name][k] != (None, None))
        judge_ranges = sum(w(k) for k in expected if expected[k] != (None, None))
        correct_ranges = sum(w(k) for k in expected if got[name][k] == expected[k] != (None, None))
        cells = (correct / total, correct_ranges / produced, correct_ranges / judge_ranges)
        line += f" {correct:>8} {total - correct:>6}   {cells[0]:9.1%} {cells[1]:10.1%} {cells[2]:7.1%}" if w(next(iter(expected))) == 1 and w is not weight else f"   {cells[0]:12.1%} {cells[1]:6.1%} {cells[2]:7.1%}"
    print(line)

9999 judged, 0 malformed lines skipped, 11 dropped as invalid

parser    correct  wrong    accuracy  precision  recall   weighted acc   prec  recall
python       4772   5279       47.5%      48.7%   48.1%          92.6%  94.9%   92.7%
scala        8443   1608       84.0%      97.2%   84.4%          96.5%  99.7%   96.5%
new          9793    258       97.4%      97.7%   98.2%          99.8%  99.8%   99.8%


In [7]:
PARSER = "new"
LIMIT = 40

for key in sorted((k for k in expected if got[PARSER][k] != expected[k]), key=lambda k: -weight(k))[:LIMIT]:
    id_, path = key
    outcome, start, end, qualifier, note = judged[id_]
    print(f"{index[id_]['text']!r}  [{path}, {weight(key)}x]")
    print(f"    judge   {show(expected[key]) if outcome == 'range' else outcome}{'  ' + repr(qualifier) if qualifier and qualifier != 'exact' else ''}{'  ' + note if note else ''}")
    print(f"    {PARSER:7} {show(got[PARSER][key])}")

'To 1763 (New France)'  [subject, 64x]
    judge   open .. 1763-12-31  'before'  parenthetical text ignored
    new     1763-01-01 .. 1763-12-31
'c.1977-1985'  [production, 63x]
    judge   1968-01-01 .. 1985-12-31  'approximate'
    new     1967-01-01 .. 1985-12-31
'Empire, 30 B.C.-284 A.D.'  [subject, 38x]
    judge   -0030-01-0 .. 0284-12-31  name ignored
    new     no range
'--1797.'  [production, 16x]
    judge   open .. 1797-12-31  'before'
    new     1797-01-01 .. 1797-12-31
'[199]'  [production, 15x]
    judge   1990-01-01 .. 1999-12-31  'approximate'  placeholder decade
    new     0199-01-01 .. 0199-12-31
'--1796.'  [production, 9x]
    judge   open .. 1796-12-31  'before'  double hyphen treated as before marker
    new     1796-01-01 .. 1796-12-31
'Han dynasty, 202 B.C.-220 A.D.'  [subject, 5x]
    judge   -0202-01-0 .. 0220-12-31  name ignored
    new     no range
'An XI -- 1803.'  [production, 5x]
    judge   ambiguous  An XI (French Republican calendar) not convertible;

## Against the 008

Sierra production dates paired with the record's own 008 range (from `period_sierra_extract`), compared
at year level since the 008 has no months. Only strings the parsers saw in `periods.jsonl` are scored.

In [8]:
def bounds_008(value):
    if value is None:
        return None
    start, sep, end = value.partition("-")
    return (start or None, (end or None) if sep else start)


def years_of(b):
    return None if b == (None, None) else tuple(d and d[:4] for d in b)


pairs = [json.loads(line) for line in Path("data/sierra-008.jsonl").open(encoding="utf-8")]
scored = Counter()
mismatches = defaultdict(Counter)
for row in pairs:
    key = ("marc", "production", row["text"])
    gold = bounds_008(row["range_008"])
    if gold is None or key not in occurrences:
        continue
    for name in PARSERS:
        got = years_of(results[name][key])
        outcome = "agree" if got == gold else "no range" if got is None else "disagree"
        scored[(name, outcome)] += row["n"]
        if outcome != "agree":
            mismatches[name][(row["text"], row["range_008"], show(results[name][key]))] += row["n"]

total = sum(n for (name, _), n in scored.items() if name == "new")
print(f"{total} production strings with an 008 range\n")
print(f"{'parser':8} {'agree':>8} {'no range':>9} {'disagree':>9}")
for name in PARSERS:
    print(f"{name:8} {scored[(name, 'agree')] / total:8.1%} {scored[(name, 'no range')] / total:9.1%} {scored[(name, 'disagree')] / total:9.1%}")

print("\nnew parser vs 008, heaviest mismatches")
for (text, range_008, got), n in mismatches["new"].most_common(30):
    print(f"{n:6}  {text!r:40} 008 {range_008:12} new {got}")

1060349 production strings with an 008 range

parser      agree  no range  disagree
python      96.3%      0.4%      3.3%
scala       95.4%      1.9%      2.6%
new         97.0%      0.1%      2.9%

new parser vs 008, heaviest mismatches
   276  '[20 ]'                                  008              new 2000-01-01 .. 2099-12-31
   275  '[201 ]'                                 008              new 2010-01-01 .. 2019-12-31
   269  '1970?'                                  008 1960-1980    new 1970-01-01 .. 1970-12-31
   248  '1970s-1980s'                            008 1970-1980    new 1970-01-01 .. 1989-12-31
   217  '[200 ]'                                 008              new 2000-01-01 .. 2009-12-31
   206  '[approximately 1900]'                   008 1900         new 1890-01-01 .. 1909-12-31
   171  '[between 1800 and 1899?]'               008              new 1800-01-01 .. 1899-12-31
   159  '1990?'                                  008 1980-2000    new 1990-01-01 .. 1990-12-31
  